# Compare QL-ABS vs Q-Learning Performance

This notebook compares QL-ABS (also referred to in the repo as Smart Q-Learning/two-stage mode) with standard Q-Learning.
For each configuration, it shows log-mean-relative-error (MRE) curves, per-state MRE, and per-state value-function convergence.

## Contents

**With Reinfection (SIRS)**
- [N = 5 — SIRS](#N-5-SIRS)
- [N = 15 — SIRS](#N-15-SIRS)

**No Reinfection (SIR)**
- [N = 5 — SIR](#N-5-SIR)
- [N = 15 — SIR](#N-15-SIR)
- [N = 50 — SIR](#N-50-SIR)

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pathlib import Path

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from tqdm import tqdm

%matplotlib inline

# Move to root folder for imports
os.chdir(Path.cwd().parent)
from src.azureml_utils import download_runs_artifacts
from experiments.utils import (
    fetch_metrics_for_runs,
    compute_mean_and_ci,
    plot_mean_mre,
    plot_mean_mre_multi_ka,
    load_run_state_value_errors,
    load_runs_state_value_errors,
    compute_per_state_series,
    plot_per_state_error,
    plot_per_state_error_multi_ka,
    load_run_value_functions,
    load_runs_value_functions,
    compute_per_state_value_series,
    plot_per_state_value_multi_ka,
    validate_runs,
)

load_dotenv()

FIGURES_DIR = Path("experiments/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Set to True to synchronise y-axis limits across states within each scenario
ALIGN_YLIM = False


def _save_fig(fig, stem: str) -> None:
    """Save *fig* as PDF to FIGURES_DIR."""
    out = FIGURES_DIR / f"{stem}.pdf"
    fig.savefig(out, bbox_inches="tight")
    print(f"Saved: {out}")


def _sync_ylim(*figs) -> None:
    """Synchronise y-axis limits across the primary axes of the given figures.

    After calling this function all axes share the union of their original
    y-ranges, making cross-condition comparisons visually consistent.
    """
    axes = [fig.axes[0] for fig in figs if fig.axes]
    lo = min(ax.get_ylim()[0] for ax in axes)
    hi = max(ax.get_ylim()[1] for ax in axes)
    for ax in axes:
        ax.set_ylim(lo, hi)
    for fig in figs:
        fig.canvas.draw_idle()


In [ ]:
try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")
    ml_client = MLClient.from_config(credential=credential)
    print(f"Connected to workspace: {ml_client.workspace_name}")
    print(f"Subscription:   {ml_client.subscription_id}")
    print(f"Resource group: {ml_client.resource_group_name}")
except Exception as e:
    print(f"Error connecting to workspace: {e}")
    raise

tracking_uri = ml_client.workspaces.get(ml_client.workspace_name).mlflow_tracking_uri
mlflow.set_tracking_uri(tracking_uri)
print(f"✓ MLflow tracking URI set to {mlflow.get_tracking_uri()}")

client = mlflow.tracking.MlflowClient()
experiment = mlflow.get_experiment_by_name("SIRS-Q-Learning")


---

## Part 1 — With Reinfection (SIRS)

### N = 5 — SIRS <a id='N-5-SIRS'></a>

In [ ]:
# ── Load & filter runs ────────────────────────────────────────────────────
runs_5_all = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.run_group = 'q_learning_5' OR tags.run_group = 'smart_5'",
    order_by=["start_time DESC"],
)

stage2_extra_steps_filter_5 = "1"   # set to None to include all two_stage runs
selected_ka_5 = ["5000", "35000", "75000"]           # set to None to plot all

runs_5 = runs_5_all[
    (runs_5_all["tags.learn_mode"] == "complete")
    | (
        (runs_5_all["tags.learn_mode"] == "two_stages")
        & (stage2_extra_steps_filter_5 is None or runs_5_all["params.stage2_absorbing_extra_steps"] == stage2_extra_steps_filter_5)
        & (selected_ka_5 is None or runs_5_all["params.first_stage_steps"].isin(selected_ka_5))
    )
]

# ── Fetch metrics & build per-K^a series ──────────────────────────────────
metrics_5 = fetch_metrics_for_runs(client, runs_5, "log_mean_relative_error_V")
complete_5 = metrics_5[metrics_5["tags.learn_mode"] == "complete"]


In [ ]:
# ── Build K^a series & plot ─────────────────────────────────────────────────
two_stage_runs_5 = runs_5[runs_5["tags.learn_mode"] == "two_stages"]
available_ka_5 = sorted(two_stage_runs_5["params.first_stage_steps"].dropna().unique(), key=int)
ka_to_plot_5 = selected_ka_5 if selected_ka_5 is not None else available_ka_5

two_stage_series_5 = []
for ka in ka_to_plot_5:
    run_ids = two_stage_runs_5[two_stage_runs_5["params.first_stage_steps"] == ka]["run_id"]
    df = metrics_5[metrics_5["run_id"].isin(run_ids)]
    if not df.empty:
        two_stage_series_5.append((f"K^a = {int(ka):,}", df))

print(f"N=5 SIRS — Q-Learning: {complete_5['run_id'].nunique()} runs")
for label, df in two_stage_series_5:
    print(f"  QL-ABS ({label}): {df['run_id'].nunique()} runs")
validate_runs(runs_5, expected_n=5, reinfection=True, label="N=5 SIRS")

# ── Mean MRE plot ─────────────────────────────────────────────────────────
fig_mre_5, _ = plot_mean_mre_multi_ka(
    complete_5, two_stage_series_5, title=None, y_label="Log Mean Relative Error (V)"
)
plt.show()
_save_fig(fig_mre_5, "ql_abs_5")


In [ ]:
# ── Download artifacts (skip already-downloaded) ──────────────────────────
download_runs_artifacts(
    run_names=runs_5["run_id"].tolist(),
    output_dir="artifacts",
    storage_account_name=os.getenv("STORAGE_ACCOUNT_NAME"),
    storage_account_key=os.getenv("STORAGE_ACCOUNT_KEY"),
    overwrite=False,
    skip_downloaded=True,
)


In [ ]:
# ── Load per-state value errors ───────────────────────────────────────────
state_errors_5 = load_runs_state_value_errors(runs_5)

n_c = sum(1 for v in state_errors_5.values() if v["learn_mode"] == "complete")
n_t = sum(1 for v in state_errors_5.values() if v["learn_mode"] == "two_stages")
print(f"Loaded {len(state_errors_5)} runs ({n_c} complete, {n_t} QL-ABS)")


In [ ]:
# ── Per-state RE plots (N=5 SIRS) ───────────────────────────────────────
# States: (m_s, m_i) with m_s + m_i <= 5
states_to_plot_5 = [(2, 2), (1, 0)]

complete_ps_5 = {rid: v for rid, v in state_errors_5.items() if v["learn_mode"] == "complete"}
two_stage_ps_series_5 = []
for ka in ka_to_plot_5:
    run_ids = set(two_stage_runs_5[two_stage_runs_5["params.first_stage_steps"] == ka]["run_id"])
    errs = {rid: v for rid, v in state_errors_5.items() if rid in run_ids}
    if errs:
        two_stage_ps_series_5.append((f"K^a = {int(ka):,}", errs))

# Plot all states, collect figure handles, then sync y-axes within this scenario
_figs = {}
for state in states_to_plot_5:
    fig, _ = plot_per_state_error_multi_ka(
        complete_ps_5,
        two_stage_ps_series_5,
        state,
        show_title=False,
        y_label="Log Relative Error (V)",
        suptitle_suffix="Per-State $\\log_{10}$ RE (V)",
    )
    _figs[state] = fig

if ALIGN_YLIM:
    _sync_ylim(*_figs.values())

for state, fig in _figs.items():
    plt.figure(fig.number)  # ensure correct figure is active for display
    plt.show()
    _save_fig(fig, f"ql_abs_5_state_{state[0]}_{state[1]}")


In [ ]:
# ── Load value functions (N=5 SIRS) ──────────────────────────────────────
vf_5 = load_runs_value_functions(runs_5)

complete_vf_5 = {rid: v for rid, v in vf_5.items() if v["learn_mode"] == "complete"}
two_stage_vf_series_5 = []
for ka in ka_to_plot_5:
    run_ids = set(two_stage_runs_5[two_stage_runs_5["params.first_stage_steps"] == ka]["run_id"])
    vfs = {rid: v for rid, v in vf_5.items() if rid in run_ids}
    if vfs:
        two_stage_vf_series_5.append((f"K^a = {int(ka):,}", vfs))

print(f"Loaded {len(vf_5)} VF runs ({len(complete_vf_5)} complete, {len(two_stage_vf_series_5)} K^a groups)")


In [ ]:
# ── Per-state value function plot (N=5 SIRS) ──────────────────────────────
plot_per_state_value_multi_ka(
    complete_vf_5,
    two_stage_vf_series_5,
    states_to_plot_5,
    size_label="N=5, SIRS",
    log_values=False,
)
plt.show()


### N = 15 — SIRS <a id='N-15-SIRS'></a>

In [ ]:
# ── Load & filter runs ────────────────────────────────────────────────────
runs_15_all = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.run_group = 'q_learning_15' OR tags.run_group = 'smart_15'",
    order_by=["start_time DESC"],
)

stage2_extra_steps_filter_15 = "1"
selected_ka_15 = ["50000", "150000", "300000", "500000"]  # set to None to plot all

runs_15 = runs_15_all[
    (runs_15_all["tags.learn_mode"] == "complete")
    | (
        (runs_15_all["tags.learn_mode"] == "two_stages")
        & (stage2_extra_steps_filter_15 is None or runs_15_all["params.stage2_absorbing_extra_steps"] == stage2_extra_steps_filter_15)
        & (selected_ka_15 is None or runs_15_all["params.first_stage_steps"].isin(selected_ka_15))
    )
]

metrics_15 = fetch_metrics_for_runs(client, runs_15, "log_mean_relative_error_V")
complete_15 = metrics_15[metrics_15["tags.learn_mode"] == "complete"]


In [ ]:
# ── Build K^a series & plot ─────────────────────────────────────────────────
two_stage_runs_15 = runs_15[runs_15["tags.learn_mode"] == "two_stages"]
available_ka_15 = sorted(two_stage_runs_15["params.first_stage_steps"].dropna().unique(), key=int)
ka_to_plot_15 = selected_ka_15 if selected_ka_15 is not None else available_ka_15

two_stage_series_15 = []
for ka in ka_to_plot_15:
    run_ids = two_stage_runs_15[two_stage_runs_15["params.first_stage_steps"] == ka]["run_id"]
    df = metrics_15[metrics_15["run_id"].isin(run_ids)]
    if not df.empty:
        two_stage_series_15.append((f"K^a = {int(ka):,}", df))

print(f"N=15 SIRS — Q-Learning: {complete_15['run_id'].nunique()} runs")
for label, df in two_stage_series_15:
    print(f"  QL-ABS ({label}): {df['run_id'].nunique()} runs")
validate_runs(runs_15, expected_n=15, reinfection=True, label="N=15 SIRS")

fig_mre_15, _ = plot_mean_mre_multi_ka(
    complete_15, two_stage_series_15, title=None, y_label="Log Mean Relative Error (V)"
)
plt.show()
_save_fig(fig_mre_15, "ql_abs_15")


In [ ]:
download_runs_artifacts(
    run_names=runs_15["run_id"].tolist(),
    output_dir="artifacts",
    storage_account_name=os.getenv("STORAGE_ACCOUNT_NAME"),
    storage_account_key=os.getenv("STORAGE_ACCOUNT_KEY"),
    overwrite=False,
    skip_downloaded=True,
)


In [ ]:
state_errors_15 = load_runs_state_value_errors(runs_15)

n_c = sum(1 for v in state_errors_15.values() if v["learn_mode"] == "complete")
n_t = sum(1 for v in state_errors_15.values() if v["learn_mode"] == "two_stages")
print(f"Loaded {len(state_errors_15)} runs ({n_c} complete, {n_t} QL-ABS)")


In [ ]:
# ── Per-state RE plots (N=15 SIRS) ───────────────────────────────────────
# States: (m_s, m_i) with m_s + m_i <= 15
states_to_plot_15 = [(5, 8), (8, 0)]

complete_ps_15 = {rid: v for rid, v in state_errors_15.items() if v["learn_mode"] == "complete"}
two_stage_ps_series_15 = []
for ka in ka_to_plot_15:
    run_ids = set(two_stage_runs_15[two_stage_runs_15["params.first_stage_steps"] == ka]["run_id"])
    errs = {rid: v for rid, v in state_errors_15.items() if rid in run_ids}
    if errs:
        two_stage_ps_series_15.append((f"K^a = {int(ka):,}", errs))

# Plot all states, collect figure handles, then sync y-axes within this scenario
_figs = {}
for state in states_to_plot_15:
    fig, _ = plot_per_state_error_multi_ka(
        complete_ps_15,
        two_stage_ps_series_15,
        state,
        show_title=False,
        y_label="Log Relative Error (V)",
        suptitle_suffix="Per-State $\\log_{10}$ RE (V)",
    )
    _figs[state] = fig

if ALIGN_YLIM:
    _sync_ylim(*_figs.values())

for state, fig in _figs.items():
    plt.figure(fig.number)  # ensure correct figure is active for display
    plt.show()
    _save_fig(fig, f"ql_abs_15_state_{state[0]}_{state[1]}")


In [ ]:
vf_15 = load_runs_value_functions(runs_15)

complete_vf_15 = {rid: v for rid, v in vf_15.items() if v["learn_mode"] == "complete"}
two_stage_vf_series_15 = []
for ka in ka_to_plot_15:
    run_ids = set(two_stage_runs_15[two_stage_runs_15["params.first_stage_steps"] == ka]["run_id"])
    vfs = {rid: v for rid, v in vf_15.items() if rid in run_ids}
    if vfs:
        two_stage_vf_series_15.append((f"K^a = {int(ka):,}", vfs))

print(f"Loaded {len(vf_15)} VF runs ({len(complete_vf_15)} complete, {len(two_stage_vf_series_15)} K^a groups)")


In [ ]:
plot_per_state_value_multi_ka(
    complete_vf_15,
    two_stage_vf_series_15,
    states_to_plot_15,
    size_label="N=15, SIRS",
)
plt.show()


---

## Part 2 — No Reinfection (SIR)

### N = 5 — SIR <a id='N-5-SIR'></a>

In [ ]:
runs_5_sir_all = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.run_group = 'q_learning_5_noreinf' OR tags.run_group = 'smart_5_noreinf'",
    order_by=["start_time DESC"],
)

stage2_extra_steps_filter_5_sir = "1"
selected_ka_5_sir = None  # set to None to plot all

runs_5_sir = runs_5_sir_all[
    (runs_5_sir_all["tags.learn_mode"] == "complete")
    | (
        (runs_5_sir_all["tags.learn_mode"] == "two_stages")
        & (stage2_extra_steps_filter_5_sir is None or runs_5_sir_all["params.stage2_absorbing_extra_steps"] == stage2_extra_steps_filter_5_sir)
        & (selected_ka_5_sir is None or runs_5_sir_all["params.first_stage_steps"].isin(selected_ka_5_sir))
    )
]

metrics_5_sir = fetch_metrics_for_runs(client, runs_5_sir, "log_mean_relative_error_V")
complete_5_sir = metrics_5_sir[metrics_5_sir["tags.learn_mode"] == "complete"]


In [ ]:
# ── Build K^a series & plot ─────────────────────────────────────────────────
two_stage_runs_5_sir = runs_5_sir[runs_5_sir["tags.learn_mode"] == "two_stages"]
available_ka_5_sir = sorted(two_stage_runs_5_sir["params.first_stage_steps"].dropna().unique(), key=int)
ka_to_plot_5_sir = selected_ka_5_sir if selected_ka_5_sir is not None else available_ka_5_sir

two_stage_series_5_sir = []
for ka in ka_to_plot_5_sir:
    run_ids = two_stage_runs_5_sir[two_stage_runs_5_sir["params.first_stage_steps"] == ka]["run_id"]
    df = metrics_5_sir[metrics_5_sir["run_id"].isin(run_ids)]
    if not df.empty:
        two_stage_series_5_sir.append((f"K^a = {int(ka):,}", df))

print(f"N=5 SIR — Q-Learning: {complete_5_sir['run_id'].nunique()} runs")
for label, df in two_stage_series_5_sir:
    print(f"  QL-ABS ({label}): {df['run_id'].nunique()} runs")
validate_runs(runs_5_sir, expected_n=5, reinfection=False, label="N=5 SIR")

fig_mre_5_sir, _ = plot_mean_mre_multi_ka(
    complete_5_sir, two_stage_series_5_sir, title=None, y_label="Log Mean Relative Error (V)"
)
plt.show()
_save_fig(fig_mre_5_sir, "ql_abs_5_noreinf")


In [ ]:
download_runs_artifacts(
    run_names=runs_5_sir["run_id"].tolist(),
    output_dir="artifacts",
    storage_account_name=os.getenv("STORAGE_ACCOUNT_NAME"),
    storage_account_key=os.getenv("STORAGE_ACCOUNT_KEY"),
    overwrite=False,
    skip_downloaded=True,
)


In [ ]:
state_errors_5_sir = load_runs_state_value_errors(runs_5_sir)

n_c = sum(1 for v in state_errors_5_sir.values() if v["learn_mode"] == "complete")
n_t = sum(1 for v in state_errors_5_sir.values() if v["learn_mode"] == "two_stages")
print(f"Loaded {len(state_errors_5_sir)} runs ({n_c} complete, {n_t} QL-ABS)")


In [ ]:
# ── Per-state RE plots (N=5 SIR) ────────────────────────────────────────
# States: (m_s, m_i) with m_s + m_i <= 5 (no reinfection) — same as SIRS
states_to_plot_5_sir = [(2, 2), (1, 0)]

complete_ps_5_sir = {rid: v for rid, v in state_errors_5_sir.items() if v["learn_mode"] == "complete"}
two_stage_ps_series_5_sir = []
for ka in ka_to_plot_5_sir:
    run_ids = set(two_stage_runs_5_sir[two_stage_runs_5_sir["params.first_stage_steps"] == ka]["run_id"])
    errs = {rid: v for rid, v in state_errors_5_sir.items() if rid in run_ids}
    if errs:
        two_stage_ps_series_5_sir.append((f"K^a = {int(ka):,}", errs))

# Plot all states, collect figure handles, then sync y-axes within this scenario
_figs = {}
for state in states_to_plot_5_sir:
    fig, _ = plot_per_state_error_multi_ka(
        complete_ps_5_sir,
        two_stage_ps_series_5_sir,
        state,
        show_title=False,
        y_label="Log Relative Error (V)",
        suptitle_suffix="Per-State $\\log_{10}$ RE (V)",
    )
    _figs[state] = fig

if ALIGN_YLIM:
    _sync_ylim(*_figs.values())

for state, fig in _figs.items():
    plt.figure(fig.number)  # ensure correct figure is active for display
    plt.show()
    _save_fig(fig, f"ql_abs_5_noreinf_state_{state[0]}_{state[1]}")


In [ ]:
vf_5_sir = load_runs_value_functions(runs_5_sir)

complete_vf_5_sir = {rid: v for rid, v in vf_5_sir.items() if v["learn_mode"] == "complete"}
two_stage_vf_series_5_sir = []
for ka in ka_to_plot_5_sir:
    run_ids = set(two_stage_runs_5_sir[two_stage_runs_5_sir["params.first_stage_steps"] == ka]["run_id"])
    vfs = {rid: v for rid, v in vf_5_sir.items() if rid in run_ids}
    if vfs:
        two_stage_vf_series_5_sir.append((f"K^a = {int(ka):,}", vfs))

print(f"Loaded {len(vf_5_sir)} VF runs ({len(complete_vf_5_sir)} complete, {len(two_stage_vf_series_5_sir)} K^a groups)")


In [ ]:
plot_per_state_value_multi_ka(
    complete_vf_5_sir,
    two_stage_vf_series_5_sir,
    states_to_plot_5_sir,
    size_label="N=5, SIR",
    log_values=False,
)
plt.show()


### N = 15 — SIR <a id='N-15-SIR'></a>

In [ ]:
runs_15_sir_all = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.run_group = 'q_learning_15_noreinf' OR tags.run_group = 'smart_15_noreinf'",
    order_by=["start_time DESC"],
)

stage2_extra_steps_filter_15_sir = "1"
selected_ka_15_sir = ["10000", "20000", "50000", "100000"] # set to a list e.g. ["50000", "100000"] to restrict

runs_15_sir = runs_15_sir_all[
    (runs_15_sir_all["tags.learn_mode"] == "complete")
    | (
        (runs_15_sir_all["tags.learn_mode"] == "two_stages")
        & (stage2_extra_steps_filter_15_sir is None or runs_15_sir_all["params.stage2_absorbing_extra_steps"] == stage2_extra_steps_filter_15_sir)
        & (selected_ka_15_sir is None or runs_15_sir_all["params.first_stage_steps"].isin(selected_ka_15_sir))
    )
]

metrics_15_sir = fetch_metrics_for_runs(client, runs_15_sir, "log_mean_relative_error_V")
complete_15_sir = metrics_15_sir[metrics_15_sir["tags.learn_mode"] == "complete"]


In [ ]:
# ── Build K^a series & plot ─────────────────────────────────────────────────
two_stage_runs_15_sir = runs_15_sir[runs_15_sir["tags.learn_mode"] == "two_stages"]
available_ka_15_sir = sorted(two_stage_runs_15_sir["params.first_stage_steps"].dropna().unique(), key=int)
ka_to_plot_15_sir = selected_ka_15_sir if selected_ka_15_sir is not None else available_ka_15_sir

two_stage_series_15_sir = []
for ka in ka_to_plot_15_sir:
    run_ids = two_stage_runs_15_sir[two_stage_runs_15_sir["params.first_stage_steps"] == ka]["run_id"]
    df = metrics_15_sir[metrics_15_sir["run_id"].isin(run_ids)]
    if not df.empty:
        two_stage_series_15_sir.append((f"K^a = {int(ka):,}", df))

print(f"N=15 SIR — Q-Learning: {complete_15_sir['run_id'].nunique()} runs")
for label, df in two_stage_series_15_sir:
    print(f"  QL-ABS ({label}): {df['run_id'].nunique()} runs")
validate_runs(runs_15_sir, expected_n=15, reinfection=False, label="N=15 SIR")

fig_mre_15_sir, _ = plot_mean_mre_multi_ka(
    complete_15_sir, two_stage_series_15_sir, title=None, y_label="Log Mean Relative Error (V)"
)
plt.show()
_save_fig(fig_mre_15_sir, "ql_abs_15_noreinf")


In [ ]:
download_runs_artifacts(
    run_names=runs_15_sir["run_id"].tolist(),
    output_dir="artifacts",
    storage_account_name=os.getenv("STORAGE_ACCOUNT_NAME"),
    storage_account_key=os.getenv("STORAGE_ACCOUNT_KEY"),
    overwrite=False,
    skip_downloaded=True,
)


In [ ]:
state_errors_15_sir = load_runs_state_value_errors(runs_15_sir)

n_c = sum(1 for v in state_errors_15_sir.values() if v["learn_mode"] == "complete")
n_t = sum(1 for v in state_errors_15_sir.values() if v["learn_mode"] == "two_stages")
print(f"Loaded {len(state_errors_15_sir)} runs ({n_c} complete, {n_t} QL-ABS)")


In [ ]:
# ── Per-state RE plots (N=15 SIR) ───────────────────────────────────────
# States: (m_s, m_i) with m_s + m_i <= 15 (no reinfection)
states_to_plot_15_sir = [(5, 8), (8, 0)]

complete_ps_15_sir = {rid: v for rid, v in state_errors_15_sir.items() if v["learn_mode"] == "complete"}
two_stage_ps_series_15_sir = []
for ka in ka_to_plot_15_sir:
    run_ids = set(two_stage_runs_15_sir[two_stage_runs_15_sir["params.first_stage_steps"] == ka]["run_id"])
    errs = {rid: v for rid, v in state_errors_15_sir.items() if rid in run_ids}
    if errs:
        two_stage_ps_series_15_sir.append((f"K^a = {int(ka):,}", errs))

# Plot all states, collect figure handles, then sync y-axes within this scenario
_figs = {}
for state in states_to_plot_15_sir:
    fig, _ = plot_per_state_error_multi_ka(
        complete_ps_15_sir,
        two_stage_ps_series_15_sir,
        state,
        show_title=False,
        y_label="Log Relative Error (V)",
        suptitle_suffix="Per-State $\\log_{10}$ RE (V)",
    )
    _figs[state] = fig

if ALIGN_YLIM:
    _sync_ylim(*_figs.values())

for state, fig in _figs.items():
    plt.figure(fig.number)  # ensure correct figure is active for display
    plt.show()
    _save_fig(fig, f"ql_abs_15_noreinf_state_{state[0]}_{state[1]}")


In [ ]:
vf_15_sir = load_runs_value_functions(runs_15_sir)

complete_vf_15_sir = {rid: v for rid, v in vf_15_sir.items() if v["learn_mode"] == "complete"}
two_stage_vf_series_15_sir = []
for ka in ka_to_plot_15_sir:
    run_ids = set(two_stage_runs_15_sir[two_stage_runs_15_sir["params.first_stage_steps"] == ka]["run_id"])
    vfs = {rid: v for rid, v in vf_15_sir.items() if rid in run_ids}
    if vfs:
        two_stage_vf_series_15_sir.append((f"K^a = {int(ka):,}", vfs))

print(f"Loaded {len(vf_15_sir)} VF runs ({len(complete_vf_15_sir)} complete, {len(two_stage_vf_series_15_sir)} K^a groups)")


In [ ]:
plot_per_state_value_multi_ka(
    complete_vf_15_sir,
    two_stage_vf_series_15_sir,
    states_to_plot_15_sir,
    size_label="N=15, SIR",
)
plt.show()


### N = 50 — SIR <a id='N-50-SIR'></a>

In [ ]:
runs_50_sir_all = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.run_group = 'q_learning_50_noreinf' OR tags.run_group = 'smart_50_noreinf'",
    order_by=["start_time DESC"],
)

stage2_extra_steps_filter_50_sir = "1"
selected_ka_50_sir = ["10000", "50000", "100000", "150000", "300000"]  # set to None to plot all

runs_50_sir = runs_50_sir_all[
    (runs_50_sir_all["tags.learn_mode"] == "complete")
    | (
        (runs_50_sir_all["tags.learn_mode"] == "two_stages")
        & (stage2_extra_steps_filter_50_sir is None or runs_50_sir_all["params.stage2_absorbing_extra_steps"] == stage2_extra_steps_filter_50_sir)
        & (selected_ka_50_sir is None or runs_50_sir_all["params.first_stage_steps"].isin(selected_ka_50_sir))
    )
]

metrics_50_sir = fetch_metrics_for_runs(client, runs_50_sir, "log_mean_relative_error_V")
complete_50_sir = metrics_50_sir[metrics_50_sir["tags.learn_mode"] == "complete"]


In [ ]:
# ── Build K^a series & plot ─────────────────────────────────────────────────
two_stage_runs_50_sir = runs_50_sir[runs_50_sir["tags.learn_mode"] == "two_stages"]
available_ka_50_sir = sorted(two_stage_runs_50_sir["params.first_stage_steps"].dropna().unique(), key=int)
ka_to_plot_50_sir = selected_ka_50_sir if selected_ka_50_sir is not None else available_ka_50_sir

two_stage_series_50_sir = []
for ka in ka_to_plot_50_sir:
    run_ids = two_stage_runs_50_sir[two_stage_runs_50_sir["params.first_stage_steps"] == ka]["run_id"]
    df = metrics_50_sir[metrics_50_sir["run_id"].isin(run_ids)]
    if not df.empty:
        two_stage_series_50_sir.append((f"K^a = {int(ka):,}", df))

print(f"N=50 SIR — Q-Learning: {complete_50_sir['run_id'].nunique()} runs")
for label, df in two_stage_series_50_sir:
    print(f"  QL-ABS ({label}): {df['run_id'].nunique()} runs")
validate_runs(runs_50_sir, expected_n=50, reinfection=False, label="N=50 SIR")

fig_mre_50_sir, _ = plot_mean_mre_multi_ka(
    complete_50_sir, two_stage_series_50_sir, title=None, y_label="Log Mean Relative Error (V)"
)
plt.show()
_save_fig(fig_mre_50_sir, "ql_abs_50_noreinf")


In [ ]:
download_runs_artifacts(
    run_names=runs_50_sir["run_id"].tolist(),
    output_dir="artifacts",
    storage_account_name=os.getenv("STORAGE_ACCOUNT_NAME"),
    storage_account_key=os.getenv("STORAGE_ACCOUNT_KEY"),
    overwrite=False,
    skip_downloaded=True,
)


In [ ]:
state_errors_50_sir = load_runs_state_value_errors(runs_50_sir)

n_c = sum(1 for v in state_errors_50_sir.values() if v["learn_mode"] == "complete")
n_t = sum(1 for v in state_errors_50_sir.values() if v["learn_mode"] == "two_stages")
print(f"Loaded {len(state_errors_50_sir)} runs ({n_c} complete, {n_t} QL-ABS)")


In [ ]:
# ── Per-state RE plots (N=50 SIR) ───────────────────────────────────────
# States: (m_s, m_i) with m_s + m_i <= 50 (no reinfection)
states_to_plot_50_sir = [(10, 35), (25, 0)]

complete_ps_50_sir = {rid: v for rid, v in state_errors_50_sir.items() if v["learn_mode"] == "complete"}
two_stage_ps_series_50_sir = []
for ka in ka_to_plot_50_sir:
    run_ids = set(two_stage_runs_50_sir[two_stage_runs_50_sir["params.first_stage_steps"] == ka]["run_id"])
    errs = {rid: v for rid, v in state_errors_50_sir.items() if rid in run_ids}
    if errs:
        two_stage_ps_series_50_sir.append((f"K^a = {int(ka):,}", errs))

# Plot all states, collect figure handles, then sync y-axes within this scenario
_figs = {}
for state in states_to_plot_50_sir:
    fig, _ = plot_per_state_error_multi_ka(
        complete_ps_50_sir,
        two_stage_ps_series_50_sir,
        state,
        show_title=False,
        y_label="Log Relative Error (V)",
        suptitle_suffix="Per-State $\\log_{10}$ RE (V)",
    )
    _figs[state] = fig

if ALIGN_YLIM:
    _sync_ylim(*_figs.values())

for state, fig in _figs.items():
    plt.figure(fig.number)  # ensure correct figure is active for display
    plt.show()
    _save_fig(fig, f"ql_abs_50_noreinf_state_{state[0]}_{state[1]}")


In [ ]:
vf_50_sir = load_runs_value_functions(runs_50_sir)

complete_vf_50_sir = {rid: v for rid, v in vf_50_sir.items() if v["learn_mode"] == "complete"}
two_stage_vf_series_50_sir = []
for ka in ka_to_plot_50_sir:
    run_ids = set(two_stage_runs_50_sir[two_stage_runs_50_sir["params.first_stage_steps"] == ka]["run_id"])
    vfs = {rid: v for rid, v in vf_50_sir.items() if rid in run_ids}
    if vfs:
        two_stage_vf_series_50_sir.append((f"K^a = {int(ka):,}", vfs))

print(f"Loaded {len(vf_50_sir)} VF runs ({len(complete_vf_50_sir)} complete, {len(two_stage_vf_series_50_sir)} K^a groups)")


In [ ]:
plot_per_state_value_multi_ka(
    complete_vf_50_sir,
    two_stage_vf_series_50_sir,
    states_to_plot_50_sir,
    size_label="N=50, SIR",
)
plt.show()
